In [ ]:
import pandas as pd
import numpy as np

# 路径示例：把你们的试验表另存为 CSV 或直接 .xlsx
# 若是 Excel：
# pilot_df = pd.read_excel("data/raw/cookie_pairs_pilot.xlsx", sheet_name="pilot")
# CSV：
pilot_df = pd.read_csv("data/raw/cookie_pairs_pilot.csv")

# 只保留未排除的配对
pilot_df = pilot_df.loc[pilot_df["excluded_YN"].fillna("N").str.upper().eq("N")].copy()


In [ ]:
# 1) 如果你的 Excel 已经有 delta 列（推荐），直接用：
diffs = pilot_df["delta_spread_ratio"].astype(float).dropna().values

# 2) 如果没有 delta 列，也可以临时算：
# diffs = (pilot_df["toaster_spread_ratio"].astype(float)
#          - pilot_df["general_spread_ratio"].astype(float)).dropna().values

mean_diff = diffs.mean()
sd_diff = diffs.std(ddof=1)  # 样本标准差
n_pilot = diffs.size

mean_diff, sd_diff, n_pilot


In [ ]:
from statsmodels.stats.power import TTestPower
import math

alpha = 0.05
target_power = 0.80  # or 0.90

if sd_diff == 0 or np.isnan(sd_diff):
    raise ValueError("Pilot SD is zero or NaN; check your pilot data.")

d_z = abs(mean_diff) / sd_diff

tt = TTestPower()
n_exact = tt.solve_power(effect_size=d_z, alpha=alpha, power=target_power, alternative='two-sided')
required_pairs = math.ceil(n_exact)

d_z, n_exact, required_pairs


In [ ]:
exclusion_rate = 0.10  # 例如 10%
planning_pairs = math.ceil(required_pairs / (1 - exclusion_rate))
planning_pairs


In [ ]:
from scipy.stats import shapiro

if n_pilot >= 3:  # Shapiro 需要至少3个样本
    stat, p_shapiro = shapiro(diffs)
    p_shapiro


In [ ]:
import json, os

os.makedirs("results", exist_ok=True)
result = {
    "alpha": alpha,
    "target_power": target_power,
    "n_pilot": int(n_pilot),
    "mean_diff": float(mean_diff),
    "sd_diff": float(sd_diff),
    "effect_size_dz": float(d_z),
    "required_pairs_ceil": int(required_pairs),
    "planning_pairs_with_10pct_buffer": int(planning_pairs)
}
with open("results/power_plan_from_pilot.json", "w") as f:
    json.dump(result, f, indent=2)

result
